# 27 · Theory — The Relational Model & Relational Algebra

Everything you've done so far rests on a small, elegant body of theory from
E. F. Codd (1970). Understanding it makes SQL *click* — you stop memorizing
syntax and start reasoning about operations on sets. This is the theory engineers
are expected to know in design reviews and interviews.

In [ ]:
# ▶ Run this cell first. It loads JupySQL and connects to the SQLite database.
%load_ext sql
from sqlalchemy import create_engine
import os

# Works whether the notebook's working dir is the repo root or notebooks/
db_path = 'data/retail.db' if os.path.exists('data/retail.db') else '../data/retail.db'
engine = create_engine(f'sqlite:///{db_path}')

%config SqlMagic.autopandas = True      # results come back as pandas DataFrames
%config SqlMagic.displaycon = False
%config SqlMagic.feedback = 0
%config SqlMagic.displaylimit = 100

%sql engine
print('Connected to', db_path)

## The relational model in five words

| Formal term | What it is | SQL name |
|-------------|-----------|----------|
| **Relation** | a set of tuples | table |
| **Tuple** | one member of the set | row |
| **Attribute** | a named component of a tuple | column |
| **Domain** | the set of allowed values | data type |
| **Degree / cardinality** | number of attributes / tuples | columns / rows |

A **relation is a set**, and a set has two consequences that pure theory demands:
1. **No duplicate tuples** — every row is unique.
2. **No inherent order** — rows have no position; you only get order via
   `ORDER BY`.

SQL is a *practical* relaxation of this: by default it works on **bags**
(multisets) that *do* allow duplicates, which is why `SELECT` can return repeated
rows and you add `DISTINCT` to recover true set semantics.

## Keys — the formal hierarchy
- **Superkey:** any set of attributes that uniquely identifies a tuple.
- **Candidate key:** a *minimal* superkey (remove any column and it stops being
  unique). A table can have several.
- **Primary key:** the candidate key you choose as *the* identifier.
- **Alternate key:** the candidate keys you didn't pick (often given `UNIQUE`).
- **Foreign key:** an attribute referencing a candidate key of another (or the
  same) relation — this is what enforces **referential integrity**.

Example: in `customers`, both `customer_id` and `email` are candidate keys;
`customer_id` is the primary key and `email` is an alternate key (`UNIQUE`).

## Relational algebra → SQL

Relational algebra is the *formal set of operations* on relations. Every operator
below has a direct SQL translation. Crucially, each operator **takes relations
and returns a relation** — the **closure property** — which is exactly why you
can nest subqueries and chain CTEs.

| Algebra | Symbol | Meaning | SQL |
|---------|--------|---------|-----|
| Selection | σ (sigma) | keep rows matching a predicate | `WHERE` |
| Projection | π (pi) | keep certain columns (as a set) | `SELECT DISTINCT cols` |
| Rename | ρ (rho) | rename a relation/attribute | `AS` |
| Union | ∪ | rows in either relation | `UNION` |
| Intersection | ∩ | rows in both | `INTERSECT` |
| Difference | − | rows in first, not second | `EXCEPT` |
| Cartesian product | × | every pairing | `CROSS JOIN` |
| Join | ⋈ | product then selection | `JOIN ... ON` |
| Division | ÷ | "for all" matching | double `NOT EXISTS` |

**Selection σ** — `WHERE`:

In [ ]:
%%sql
SELECT * FROM products WHERE unit_price > 50;

**Projection π** — note `DISTINCT` for a true *set* projection:

In [ ]:
%%sql
SELECT DISTINCT category_id FROM products ORDER BY category_id;

**Cartesian product ×** — every pairing (20 products × 5 categories = 100):

In [ ]:
%%sql
SELECT COUNT(*) AS pairings FROM products CROSS JOIN categories;

**Join ⋈** — a product followed by a selection on matching keys:

In [ ]:
%%sql
SELECT p.product_name, c.category_name
FROM products p JOIN categories c USING (category_id)
LIMIT 5;

**Union / Intersection / Difference** — set operations on two relations:

In [ ]:
%%sql
SELECT country FROM customers
INTERSECT
SELECT country FROM suppliers;

## Relational division — the "for all" operator
Division answers *"find X related to **every** Y"*. There's no keyword for it;
the idiom is **double `NOT EXISTS`** ("there is no Y that X is *not* related to").

Here: employees who have handled an order of **every** status that exists.

In [ ]:
%%sql
SELECT e.employee_id, e.first_name
FROM employees e
WHERE NOT EXISTS (
    SELECT s.status
    FROM (SELECT DISTINCT status FROM orders) AS s          -- all statuses (the divisor)
    WHERE NOT EXISTS (
        SELECT 1 FROM orders o
        WHERE o.employee_id = e.employee_id
          AND o.status = s.status
    )
)
ORDER BY e.employee_id;

Read it inside-out: *keep an employee when there is no status that they have not handled.* That double-negative is the signature of relational division.

## Why this matters
Because every operation returns a relation, SQL is **compositional**: the output
of one query is valid input to another. That single property is what makes
subqueries, derived tables, CTEs, and views possible — and it's why "think in
sets, not loops" is the whole game.

## Practice

**✏️ Exercise 1.** Express in SQL the relational-algebra expression: the projection of distinct customer countries, selected to those in Europe is hard to define here — instead, give the set difference of customer countries minus supplier countries (customers-only countries).

Try it yourself in the practice cell, then run the solution to check.

In [ ]:
%%sql
-- Your turn! Replace the line below with your own query.
SELECT 'edit me, then run' AS your_answer;

<details><summary>💡 Show solution</summary>

Run the next cell to see one correct answer.</details>

In [ ]:
%%sql
SELECT country FROM customers
EXCEPT
SELECT country FROM suppliers;

### ✅ Recap
Relations are sets of tuples; keys form a hierarchy (super → candidate → primary
/ alternate / foreign); relational algebra's operators map one-to-one onto SQL;
and the closure property (relation in, relation out) is what makes SQL
composable.

**Next:** `28_theory_transactions_and_isolation.ipynb`.